In [151]:
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('..'))
from lib_equations import rk4
# import lib_plot

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [152]:
# GENERIC PARAMETERS (CHANGEABLE)
N_p = 64
mL = 8          # normalised lenght 
sm = 0.8

# POTENTIAL
def V_m(x, y, t):
	return np.zeros_like(x)

# DERIVED PARAMETERS (FIXED)
a = np.full(2, mL / N_p)	# (m)a = (m)L / N 	phyical parameter that connects lenght to resolution
				# is equal at the right and at the left
k = 1/(2 * a**2)


# SPACE
x_0 = (mL/4, mL/2)

# MOMENTUM
p_0 = (-(2*np.pi)*6, 0)

# TIME
d_tau = 0.002
max_t = 0.5

# COORDINATES
x_coo = np.linspace(0, mL, N_p)
y_coo = np.linspace(0, mL, N_p)
X, Y = np.meshgrid(x_coo, y_coo)

t_coo = np.arange(0,max_t, d_tau)

# -------------------------------------------

psi_0 = (np.exp(-((X - x_0[0])**2 + (Y - x_0[1])**2) / sm**2) * 
		 np.exp(1j * (p_0[0] * X + p_0[1] * Y))).flatten()
norm = np.sqrt(np.sum(np.abs(psi_0)**2))

psi_0 = (psi_0 / norm).flatten()

# Eq. Schr.
# psi(x, t+dt) = psi(x, t) - dt i H psi(x, t)
# --> d(psi) / dt = - i H psi(x, t)

# --> f_xy = - i (H psi(x, t))
def H_m (t, psi_flat):
	psi_mat = psi_flat.reshape((N_p, N_p))
	lamb = 2 * np.sum(k) + V_m(X, Y, t)

	# row shift
	row = ( k[0] * (np.roll(psi_mat, shift=-1, axis=0) + 
	np.roll(psi_mat, shift=1, axis=0)))
	
	# column shift
	col = (k[1] * (np.roll(psi_mat, shift=-1, axis=1) + 
	np.roll(psi_mat, shift=1, axis=1)))
	
	next_ij = row + col
	H_flat = (- next_ij + lamb * psi_mat).flatten()
	return -1j * H_flat

H_0 = H_m(0, psi_0)
psi_t = rk4(H_m, psi_0, t_coo)

In [153]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots()
im = ax.imshow((np.abs(psi_0.reshape((N_p, N_p))))**2, animated=True)

def init():
    im.set_array((np.abs(psi_0.reshape((N_p, N_p))))**2)
    return [im]

def update(frame):
	psi_mat = (np.abs(psi_t[frame].reshape((N_p, N_p))))**2
	im.set_array(psi_mat)
    
	return [im]

# frames=100 definisce la durata, interval=50 sono i millisecondi tra i frame
ani = FuncAnimation(fig, update, frames=len(psi_t), interval=10,
                     init_func=init, blit=True)

plt.close()
HTML(ani.to_jshtml())